# Build the Job Postings Knowledge Base

In this module you will build a RAG application on top of real AI Engineer job postings.

Before we can retrieve anything, we need a knowledge base. This notebook creates it in four steps:

1. Scrape recent AI Engineer job postings, like we did in module 1.
2. Use an LLM to classify whether a posting is really an AI engineering role.
3. Save every accepted posting as its own Markdown file.
4. Research every hiring company with web search and save a company profile as its own Markdown file.

Each pipeline run writes into a new folder named after the run timestamp, so you can rerun this notebook at any time without overwriting an existing knowledge base:

```
knowledge-base/
└── 2026-09-14_0938/
    ├── jobs/        one Markdown file per job posting
    └── companies/   one Markdown file per hiring company
```

In [15]:
import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from jobspy import scrape_jobs
from openai import OpenAI

In [16]:
load_dotenv(override=True)

True

## Step 1: Scrape Recent Job Postings

We search LinkedIn and Indeed for full-time AI Engineer jobs in the USA from the last 30 days.

We ask JobSpy for descriptions in Markdown format, because Markdown is exactly what we want to store in our knowledge base later.

If scraping takes too long on your machine, reduce `results_wanted` from `50` to a smaller value such as `10`.

In [17]:
jobs = scrape_jobs(
    site_name=["linkedin", "indeed"],
    linkedin_fetch_description=True,
    description_format="markdown",
    search_term='"AI Engineer"',
    location="USA",
    country_indeed="USA",
    job_type="fulltime",
    hours_old=720,  # 30 days
    results_wanted=10,
)

print(f"Total jobs scraped: {len(jobs)}")

Total jobs scraped: 20


## Clean the Scraped Jobs

We apply the same three filters as in module 1:

1. Keep only jobs that have a title, a company, a URL, and a description.
2. Keep only jobs whose title contains both `AI` and `Engineer`.
3. Remove duplicate title and company combinations.

In [18]:
jobs_df = pd.DataFrame(jobs)

# 1. Keep only jobs that have all the fields we need for the Markdown files.
required_columns = ["title", "company", "job_url", "description"]
has_required_values = (
    jobs_df[required_columns]
    .fillna("")
    .apply(lambda column: column.astype(str).str.strip() != "")
    .all(axis=1)
)
jobs_df = jobs_df[has_required_values]
print(f"Jobs with all required fields: {len(jobs_df)}")

# 2. Keep only titles that contain both "AI" and "Engineer".
# Indeed also searches descriptions, so without this we get unrelated roles.
title_contains_ai = jobs_df["title"].str.contains("AI", case=False, na=False)
title_contains_engineer = jobs_df["title"].str.contains(
    "Engineer", case=False, na=False
)
jobs_df = jobs_df[title_contains_ai & title_contains_engineer]
print(f"Jobs with 'AI' and 'Engineer' in the title: {len(jobs_df)}")

# 3. The same role often appears on several platforms.
jobs_df = jobs_df.drop_duplicates(subset=["title", "company"])
print(f"Jobs after removing duplicates: {len(jobs_df)}")

# We only keep the columns we need, and reset the index to 0, 1, 2, ...
# The reset matters because we attach the LLM classifications by position later.
jobs_df = jobs_df[
    ["title", "company", "location", "job_url", "description"]
].reset_index(drop=True)

Jobs with all required fields: 20
Jobs with 'AI' and 'Engineer' in the title: 16
Jobs after removing duplicates: 16


## Step 2: Classify the Postings With an LLM

Not every job with "AI Engineer" in the title is actually an AI engineering role.

We let an LLM decide, so that only relevant postings end up in our knowledge base.

In [19]:
client = OpenAI()

# Alternatively, if you want to use a local model 👇

# OLLAMA_BASE_URL = "http://localhost:11434/v1"

# client = OpenAI(
#    base_url=OLLAMA_BASE_URL,
#    api_key="ollama",
# )

### The Classification Instructions

The instructions tell the model what counts as an AI engineering role for this course.

In [20]:
instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models or in other words integrating them into products.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- MLOps and platform engineering are not AI engineering, as they focus on infrastructure and tooling rather than building AI-powered features.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
- In reason, briefly explain the main evidence for the decision in one sentence.
""".strip()

### The Output Schema

We ask the model for structured output: one boolean decision and one short reason.

That makes the answer easy to parse and to store next to the job data.

In [21]:
schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

### Classify Each Posting

This loop sends one LLM request per job posting.

In [22]:
results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    print(f"Classifying job {i}/{len(jobs_df)}: {job['title']}")

    response = client.responses.create(
        model="gpt-5.4-mini",  # or local model like "gemma4:e4b"
        instructions=instructions,
        input=f"""Classify this job posting.\n\nTitle: {job["title"]}\n\nDescription:\n{job["description"]}""",
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    classification = json.loads(response.output_text)
    results.append(classification)

Classifying job 1/16: Applied AI Software Engineer, GTM Growth Engineering
Classifying job 2/16: Agentic AI Engineer – Scientific Discovery
Classifying job 3/16: Founding AI Engineer
Classifying job 4/16: AI Engineer
Classifying job 5/16: AI Engineer
Classifying job 6/16: AI Engineer (Early Career)
Classifying job 7/16: Staff AI Engineer, GTM Claudification
Classifying job 8/16: AI Engineer I
Classifying job 9/16: Applied AI Engineer, Beneficial Deployments (Life Sciences)
Classifying job 10/16: AI Engineer
Classifying job 11/16: AI/ML Engineer
Classifying job 12/16: AI Engineer – Center of Excellence
Classifying job 13/16: AI/ML Engineer
Classifying job 14/16: AI/ML Engineer, Senior
Classifying job 15/16: AI/ML Engineer - Hybrid Onsite In Eden Prairie, MN
Classifying job 16/16: Generative AI & Machine Learning Engineer


### Keep Only the AI Engineering Roles

We attach the classifications to the jobs and then keep only the accepted postings.

Those are the ones that go into our knowledge base.

In [23]:
classified_jobs = pd.concat([jobs_df, pd.DataFrame(results)], axis=1)
ai_engineering_jobs = classified_jobs[classified_jobs["is_ai_engineering_role"]]

print(f"Jobs classified: {len(classified_jobs)}")
print(f"Accepted as AI engineering roles: {len(ai_engineering_jobs)}")
print(f"Rejected: {len(classified_jobs) - len(ai_engineering_jobs)}")

Jobs classified: 16
Accepted as AI engineering roles: 14
Rejected: 2


## Step 3: Write the Job Markdown Files

Now we save each accepted posting as its own Markdown file.

One job posting per file is a good starting point for RAG: the files are small enough to retrieve as a whole, and each one is about exactly one role.

Every file starts with a YAML frontmatter block. Frontmatter is metadata at the top of a Markdown file, fenced by `---` lines. Later we can use it to show the job title and link next to a retrieved answer.

The `company_file` field points to the company profile that we create in step 4.

In [24]:
def slugify(text):
    """Turn a title or company into a safe file name part: 'Senior AI Engineer' -> 'senior-ai-engineer'."""
    slug = re.sub(r"[^a-z0-9]+", "-", str(text).lower())
    return slug.strip("-")[:40]


def as_yaml_string(value):
    """Wrap a value in quotes, so the frontmatter stays valid even for titles with colons."""
    text = str(value).replace('"', "'")
    return f'"{text}"'


def build_markdown(job, scraped_at):
    """Build the Markdown content for one job posting."""
    company_file = f"companies/{slugify(job['company'])}.md"

    frontmatter = "\n".join(
        [
            "---",
            f"title: {as_yaml_string(job['title'])}",
            f"company: {as_yaml_string(job['company'])}",
            f"company_file: {as_yaml_string(company_file)}",
            f"location: {as_yaml_string(job['location'])}",
            f"job_url: {as_yaml_string(job['job_url'])}",
            f"scraped_at: {as_yaml_string(scraped_at)}",
            "---",
        ]
    )

    return f"{frontmatter}\n\n# {job['title']}\n\n{job['description']}\n"

### Create the Folders for This Pipeline Run

Each run gets its own folder named after the current date and time, for example `knowledge-base/2026-09-14_0932`.

Inside it, job postings go into `jobs/` and company profiles into `companies/`.

That way every run is a complete snapshot, and older runs stay untouched.

In [25]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
run_folder = Path("knowledge-base") / run_timestamp
jobs_folder = run_folder / "jobs"
companies_folder = run_folder / "companies"

jobs_folder.mkdir(parents=True, exist_ok=True)
companies_folder.mkdir(parents=True, exist_ok=True)

print(f"Writing knowledge base files to: {run_folder.resolve()}")

Writing knowledge base files to: /Users/lukaslechner/PythonProjects/AI-Engineering-Foundations-Labs/7-rag/knowledge-base/2026-09-14_1417


In [26]:
for i, (_, job) in enumerate(ai_engineering_jobs.iterrows(), start=1):
    # The number prefix keeps file names unique, even for two jobs with the same title.
    file_name = f"{i:02d}-{slugify(job['company'])}-{slugify(job['title'])}.md"
    file_path = jobs_folder / file_name
    file_path.write_text(build_markdown(job, run_timestamp), encoding="utf-8")

print(f"Saved {len(ai_engineering_jobs)} job postings as Markdown files.")

Saved 14 job postings as Markdown files.


### Inspect the Job Files

Let's list the job files we just created and print the beginning of the first one.

In [27]:
job_files = sorted(jobs_folder.glob("*.md"))

for file_path in job_files:
    print(f"{file_path.name} ({file_path.stat().st_size} bytes)")

01-openai-applied-ai-software-engineer-gtm-growth-.md (7015 bytes)
02-cyrad-solutions-agentic-ai-engineer-scientific-discovery.md (3269 bytes)
03-david-joseph-company-founding-ai-engineer.md (2504 bytes)
04-coreweave-ai-engineer.md (8352 bytes)
05-t12-technologies-ai-engineer.md (4640 bytes)
06-embedding-vc-ai-engineer-early-career.md (1752 bytes)
07-anthropic-staff-ai-engineer-gtm-claudification.md (9172 bytes)
08-victra-verizon-authorized-retailer-ai-engineer-i.md (5730 bytes)
09-anthropic-applied-ai-engineer-beneficial-deploymen.md (7822 bytes)
10-univera-healthcare-ai-engineer.md (8063 bytes)
11-optum-ai-ml-engineer.md (6908 bytes)
12-aderant-ai-engineer-center-of-excellence.md (3144 bytes)
13-booz-allen-hamilton-ai-ml-engineer.md (6077 bytes)
14-thomson-reuters-generative-ai-machine-learning-engineer.md (12486 bytes)


In [28]:
print(job_files[0].read_text(encoding="utf-8")[:1500])

---
title: "Applied AI Software Engineer, GTM Growth Engineering"
company: "OpenAI"
company_file: "companies/openai.md"
location: "San Francisco, CA, US"
job_url: "https://www.indeed.com/viewjob?jk=75d342ba488eccef"
scraped_at: "2026-09-14_1417"
---

# Applied AI Software Engineer, GTM Growth Engineering

Applied AI \- San Francisco

  


**About the Team**


GTM Growth Engineering builds AI\-native products that help OpenAI's go\-to\-market and B2B marketing organizations scale with greater speed, intelligence, and operational effectiveness.

  

  

We apply OpenAI models to real business workflows and build the systems that make those applications useful and dependable: customer context, agent behavior, feedback, evaluation, experimentation, and appropriate human oversight.  



  

Our work brings together software engineering, applied AI, product, data, and GTM operations. We measure success through the quality of customer engagement, pipeline, conversion, and the effectiveness of

## Step 4: Research the Hiring Companies With Web Search

A job posting tells us a lot about the role, but usually very little about the company behind it.

Could we simply ask the LLM what it knows about each company? For large companies like OpenAI or Anthropic, the model knows quite a bit, but that knowledge stops at its training cutoff. For small companies, the model often knows nothing at all and invents a plausible-sounding description instead. Invented facts in a knowledge base are dangerous, because our RAG application would later present them as the truth.

So we give the model a tool: **web search**. The Responses API has a built-in `web_search` tool. When we enable it, the model decides on its own what to search for, reads the results, and writes an answer with links to its sources.

Two things to keep in mind:

- Web search only works with the OpenAI API, not with local models.
- Web search calls are billed on top of the tokens. Check the [OpenAI pricing page](https://platform.openai.com/docs/pricing) for current prices.

### Get the Unique Companies

Some companies post several roles. We only want to research each company once.

We keep one job row per company, because the job title and location help the model find the right company when several companies have similar names.

In [29]:
companies = ai_engineering_jobs.drop_duplicates(subset="company")

print(f"Companies to research: {len(companies)}")

Companies to research: 13


### The Research Instructions

The instructions define the sections of every company profile. Using the same sections for every company keeps the knowledge base consistent.

Two rules are especially important:

- The model should say when it can't find reliable information, instead of guessing.
- The model should flag recruiting agencies that hire on behalf of an unnamed client.

In [30]:
research_instructions = """
You research companies that are hiring AI Engineers and write a short company profile in Markdown.

Use web search to find current and reliable information about the company.

Write the profile with exactly these sections:

## Overview
What the company does, where it is headquartered, and roughly how many employees it has.

## Products and Services
The main products or services of the company.

## Funding and Ownership
Whether the company is public, private, venture-backed, or part of a larger group, and notable funding rounds.

## AI Focus
How the company builds or uses AI, and what AI engineers there would likely work on.

## Sources
A bullet list of the URLs you used.

Rules:
- Only state facts that you found in your search results. If you can't find reliable information for a section, write "No reliable information found."
- Use the job title and location to identify the right company when several companies have similar names.
- If the company is a recruiting or staffing agency hiring on behalf of an unnamed client, say so clearly at the start of the Overview.
- Don't add a top-level heading. Start directly with "## Overview".
""".strip()

### Build the Company Markdown

Company files get their own frontmatter, just like the job files.

In [31]:
def build_company_markdown(company, profile, researched_at):
    """Build the Markdown content for one company profile."""
    frontmatter = "\n".join(
        [
            "---",
            f"company: {as_yaml_string(company)}",
            f"researched_at: {as_yaml_string(researched_at)}",
            "---",
        ]
    )

    return f"{frontmatter}\n\n# {company}\n\n{profile}\n"

### Research Each Company

This loop sends one LLM request per company. Compared to the classification step, there are two differences:

1. We pass `tools=[{"type": "web_search"}]`, which allows the model to search the web.
2. We don't use structured output. The model writes the Markdown profile directly, including links to the pages it found.

Each request takes a few seconds, because the model runs one or more searches before it answers.

In [32]:
for i, (_, job) in enumerate(companies.iterrows(), start=1):
    print(f"Researching company {i}/{len(companies)}: {job['company']}")

    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=research_instructions,
        input=f"""Research this company.\n\nCompany: {job["company"]}\n\nHiring for: {job["title"]} in {job["location"]}""",
        tools=[{"type": "web_search"}],
    )

    # This matches the company_file path in the job frontmatter.
    file_path = companies_folder / f"{slugify(job['company'])}.md"
    markdown = build_company_markdown(
        job["company"], response.output_text, run_timestamp
    )
    file_path.write_text(markdown, encoding="utf-8")

print(f"Saved {len(companies)} company profiles as Markdown files.")

Researching company 1/13: OpenAI
Researching company 2/13: Cyrad Solutions
Researching company 3/13: David Joseph & Company
Researching company 4/13: CoreWeave
Researching company 5/13: T12 Technologies
Researching company 6/13: Embedding VC
Researching company 7/13: Anthropic
Researching company 8/13: Victra-Verizon Authorized Retailer
Researching company 9/13: Univera Healthcare
Researching company 10/13: Optum
Researching company 11/13: Aderant
Researching company 12/13: Booz Allen Hamilton
Researching company 13/13: Thomson Reuters
Saved 13 company profiles as Markdown files.


### Inspect the Company Profiles

Let's list the company files and print the first profile.

Open a few of the source links and check whether the facts hold up. Web search makes invented facts much less likely, but not impossible.

In [33]:
company_files = sorted(companies_folder.glob("*.md"))

for file_path in company_files:
    print(f"{file_path.name} ({file_path.stat().st_size} bytes)")

aderant.md (2491 bytes)
anthropic.md (2638 bytes)
booz-allen-hamilton.md (2744 bytes)
coreweave.md (2561 bytes)
cyrad-solutions.md (2152 bytes)
david-joseph-company.md (2561 bytes)
embedding-vc.md (2602 bytes)
openai.md (2321 bytes)
optum.md (2567 bytes)
t12-technologies.md (2633 bytes)
thomson-reuters.md (2503 bytes)
univera-healthcare.md (2367 bytes)
victra-verizon-authorized-retailer.md (2280 bytes)


In [34]:
print(company_files[0].read_text(encoding="utf-8"))

---
company: "Aderant"
researched_at: "2026-09-14_1417"
---

# Aderant

## Overview
Aderant is a legal software company that provides business management and practice-of-law solutions for law firms. It is headquartered in Atlanta, Georgia, and its website states it has 700+ employees around the globe. Aderant operates as a business unit and subsidiary of Roper Technologies. ([aderant.com](https://www.aderant.com/about-aderant/?utm_source=openai))

## Products and Services
Aderant’s core offerings include business management and practice-of-law software for law firms, with products spanning financial management, matter management, billing, compliance, analytics, cloud platforms, and workflow tools. Its site highlights products and platforms such as Expert Sierra, Stridyn, and MADDI, along with newer AI agent capabilities through Agent Center. ([aderant.com](https://www.aderant.com/aderant-firm/?utm_source=openai))

## Funding and Ownership
Aderant is not publicly traded on its own; it i

Your knowledge base is ready. It contains two kinds of documents: job postings in `jobs/` and company profiles in `companies/`.

In the next notebook, we will load these Markdown files, split them into chunks, and make them searchable for our RAG application.